# PROCESAMIENTO DE TEXTOS - Dataset TFM
## Sistema de Asistencia Lectura para Discapacidad Cognitiva
**Cristina Varela - IABD**

---

**Objetivo:** Procesar archivos .txt y generar dataset ampliado con ~70-100 textos etiquetados.

**Input:** 
- 24 archivos .txt organizados en carpetas por nivel

**Output:**
- CSV con 70-100 fragmentos de texto etiquetados
- Clasificador re-entrenado con mejor accuracy

## 1. Importar Librerías

In [ ]:
import os
import re
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

print("✓ Librerías importadas correctamente")

: 

## 2. Configuración de Rutas



In [ ]:
# Ruta base del proyecto — ajustar según entorno de ejecución
BASE_DIR = Path(r'C:\Users\crisv\OneDrive\Desktop\asistente-lectura-discapacidad')

# Carpetas de textos organizadas por nivel de dificultad
# Cada carpeta contiene archivos .txt de una única fuente
CARPETAS_TEXTOS = {
    1: BASE_DIR / 'data' / 'raw' / 'nivel_1_lectura_facil',    # Plena Inclusión
    2: BASE_DIR / 'data' / 'raw' / 'nivel_2_cuentos_simple',   # Fábulas adaptadas
    3: BASE_DIR / 'data' / 'raw' / 'nivel_3_cuentos_complejo', # Cuentos complejos
    4: BASE_DIR / 'data' / 'raw' / 'nivel_4_narrativo_adulto', # Narrativa adulta
    5: BASE_DIR / 'data' / 'raw' / 'nivel_5_tecnico_especializado', # Textos técnicos
}

# CSV de salida con todos los fragmentos etiquetados
OUTPUT_CSV = BASE_DIR / 'data' / 'processed' / 'textos_etiquetados.csv'

print(f"✓ Configuración de rutas establecida")
print(f"  Base: {BASE_DIR}")
print(f"  Output: {OUTPUT_CSV}")

## 3. Funciones de Procesamiento

In [ ]:
def limpiar_texto(texto):
    """
    Elimina metadata web y ruido de los archivos .txt de origen.

    Los textos descargados de Plena Inclusión y otras fuentes web
    incluyen botones de redes sociales, fechas de publicación y
    elementos de navegación que contaminan el corpus si no se eliminan.

    Args:
        texto (str): contenido raw del archivo .txt

    Returns:
        str: texto limpio listo para fragmentar
    """
    # Cadenas de metadata web que no forman parte del contenido
    lineas_eliminar = [
        'Escuchar el texto', 'Publicado el', 'Guardado en',
        'Ver la versión', 'Twitter', 'Facebook', 'Telegram', 'WhatsApp',
    ]

    lineas = texto.split('\n')
    lineas_limpias = []

    for linea in lineas:
        linea = linea.strip()
        if len(linea) < 10:                                   # Descartar separadores y líneas vacías
            continue
        if any(meta in linea for meta in lineas_eliminar):    # Descartar metadata de redes sociales
            continue
        lineas_limpias.append(linea)

    texto_limpio = ' '.join(lineas_limpias)
    texto_limpio = re.sub(r'\s+', ' ', texto_limpio)          # Colapsar espacios múltiples

    return texto_limpio.strip()


def dividir_en_fragmentos(texto, min_palabras=25, max_palabras=50):
    """
    Divide un texto en fragmentos respetando los límites de frase.

    NOTA: Esta es la fragmentación uniforme del pipeline inicial.
    En la Iteración 2 se sustituyó por fragmentación adaptativa por nivel
    (ver codigo_fragmentacion_adaptativa.py), que mejoró el recall del
    Nivel 3 de 32% a 81% en una sola iteración.

    Args:
        texto (str): texto limpio a fragmentar
        min_palabras (int): mínimo de palabras por fragmento (default 25)
        max_palabras (int): máximo de palabras por fragmento (default 50)

    Returns:
        list[str]: lista de fragmentos que cumplen el rango de palabras
    """
    # Segmentar en frases como unidad mínima indivisible
    frases = re.split(r'[.!?]+', texto)
    frases = [f.strip() for f in frases if f.strip()]

    fragmentos = []
    fragmento_actual = []   # Frases acumuladas en el fragmento en curso
    palabras_actual = 0     # Contador de palabras del fragmento en curso

    for frase in frases:
        num_palabras = len(frase.split())

        # Si añadir esta frase supera el máximo, cerrar fragmento y empezar uno nuevo
        if palabras_actual + num_palabras > max_palabras and fragmento_actual:
            fragmentos.append(' '.join(fragmento_actual) + '.')
            fragmento_actual = [frase]
            palabras_actual = num_palabras
        else:
            # La frase cabe: añadir al fragmento en curso
            fragmento_actual.append(frase)
            palabras_actual += num_palabras

        # Cerrar fragmento al alcanzar el mínimo (requiere al menos 2 frases)
        if palabras_actual >= min_palabras and len(fragmento_actual) >= 2:
            fragmentos.append(' '.join(fragmento_actual) + '.')
            fragmento_actual = []
            palabras_actual = 0

    # Añadir el último fragmento si cumple el mínimo de palabras
    if fragmento_actual and palabras_actual >= min_palabras:
        fragmentos.append(' '.join(fragmento_actual) + '.')

    return fragmentos


def procesar_archivo(filepath, nivel):
    """
    Lee un archivo .txt y devuelve sus fragmentos etiquetados con el nivel.

    Gestiona dos encodings habituales en fuentes de texto español:
    UTF-8 (estándar) y Latin-1 (documentos más antiguos o de Windows).

    Args:
        filepath (Path): ruta al archivo .txt
        nivel (int): nivel de dificultad asignado al archivo (1-5)

    Returns:
        list[dict]: lista de registros con claves:
            texto, nivel, fuente, fragmento_num
    """
    # Intentar UTF-8 primero; fallback a Latin-1 para archivos legacy
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            texto = f.read()
    except UnicodeDecodeError:
        with open(filepath, 'r', encoding='latin-1') as f:
            texto = f.read()

    texto_limpio = limpiar_texto(texto)
    if not texto_limpio:
        return []  # Archivo vacío o sin contenido válido tras limpieza

    fragmentos = dividir_en_fragmentos(texto_limpio)

    registros = []
    for i, fragmento in enumerate(fragmentos):
        if len(fragmento.split()) >= 12:  # Filtro mínimo: descartar fragmentos de menos de 12 palabras
            registros.append({
                'texto': fragmento,
                'nivel': nivel,
                'fuente': filepath.name,
                'fragmento_num': i + 1
            })

    return registros

print("✓ Funciones de procesamiento definidas")

## 4. Procesar textos

In [ ]:
# Listas acumuladoras para todos los niveles
todos_los_textos = []
stats = {}

print("Procesando archivos...\n")

for nivel, carpeta in CARPETAS_TEXTOS.items():
    if not carpeta.exists():
        print(f"⚠️  Carpeta no encontrada (nivel {nivel}): {carpeta}")
        print(f"   Saltando...\n")
        continue
    
    # glob('*.txt') devuelve todos los archivos de texto de la carpeta
    archivos_txt = list(carpeta.glob('*.txt'))
    
    if not archivos_txt:
        print(f"⚠️  No hay archivos .txt en nivel {nivel}: {carpeta}")
        continue
    
    print(f"📁 Nivel {nivel}: {carpeta.name}")
    print(f"   Archivos encontrados: {len(archivos_txt)}")
    
    fragmentos_nivel = 0
    
    for archivo in archivos_txt:
        registros = procesar_archivo(archivo, nivel)
        # extend en lugar de append para aplanar la lista de registros
        todos_los_textos.extend(registros)
        fragmentos_nivel += len(registros)
        print(f"   ✓ {archivo.name}: {len(registros)} fragmentos")
    
    # Guardar conteo por nivel para el resumen final
    stats[nivel] = fragmentos_nivel
    print(f"   Total nivel {nivel}: {fragmentos_nivel} fragmentos\n")

print("=" * 60)
print(f"✓ PROCESAMIENTO COMPLETADO")
print(f"  Total fragmentos: {len(todos_los_textos)}")
print("\nDistribución por nivel:")
for nivel, count in sorted(stats.items()):
    print(f"  Nivel {nivel}: {count} textos")
print("=" * 60)

## 5. Crear DataFrame y Visualizar

In [ ]:
# Convertir lista de dicts a DataFrame
df = pd.DataFrame(todos_los_textos)

print(f"Dataset creado: {len(df)} textos")
print(f"\nPrimeros 5 registros:")
df.head()

In [ ]:
# Estadísticas del dataset
print("\n📊 ESTADÍSTICAS DEL DATASET")
print("=" * 60)
print(f"Total de textos: {len(df)}")
print(f"\nDistribución por nivel:")
print(df['nivel'].value_counts().sort_index())
print(f"\nLongitud promedio (palabras):")
df['num_palabras'] = df['texto'].apply(lambda x: len(x.split()))
print(df.groupby('nivel')['num_palabras'].agg(['mean', 'min', 'max']).round(1))

In [ ]:
# Visualizar distribución
import plotly.express as px
# Histograma interactivo: permite detectar desbalanceo entre niveles
fig = px.histogram(df, x='nivel', 
                   title='Distribución de Textos por Nivel de Dificultad',
                   labels={'nivel': 'Nivel de Dificultad', 'count': 'Número de Textos'},
                   color='nivel',
                   nbins=5)

fig.update_layout(showlegend=False, bargap=0.1)
fig.show()

## 6. Ver Ejemplos de Cada Nivel

In [ ]:
# NOTA: Usa 'df' completo (con columna 'fuente'), no 'df_final'
print("\n📝 EJEMPLOS DE TEXTOS POR NIVEL")
print("=" * 80)

for nivel in sorted(df['nivel'].unique()):
    ejemplo = df[df['nivel'] == nivel].iloc[0]
    print(f"\nNIVEL {nivel}:")
    print(f"Fuente: {ejemplo['fuente']}")
    print(f"Texto: {ejemplo['texto'][:200]}...")
    print("-" * 80)

## 7. Guardar Dataset

In [ ]:
# Crear carpeta processed si no existe
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

# Guardar solo texto y nivel: fuente y fragmento_num no son necesarios
# para el entrenamiento del clasificador y reducen el tamaño del CSV
df_final = df[['texto', 'nivel']].copy()

df_final.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

print(f"✓ Dataset guardado en: {OUTPUT_CSV}")
print(f"  Total de textos: {len(df_final)}")
print(f"  Tamaño del archivo: {OUTPUT_CSV.stat().st_size / 1024:.1f} KB")

## 8. Entrenar Clasificador con Nuevo Dataset

**Aquí es donde vemos la mejora en accuracy**

In [ ]:
# NOTA: contar_silabas y calcular_metricas_texto se redefinen aquí
# para que esta sección pueda ejecutarse de forma independiente
# sin depender de que la Sección 3 esté en memoria.
# Diferencia: esta versión devuelve lista directamente (no dict)
# para construir la matriz X de forma más eficiente.

def contar_silabas(palabra):
    """Cuenta sílabas aproximadas en español por grupos de vocales."""
    palabra = palabra.lower().replace('h', '')  # h muda no forma sílaba
    vocales = 'aeiouáéíóúü'
    silabas = 0
    en_vocal = False

    for char in palabra:
        if char in vocales:
            if not en_vocal:
                silabas += 1
                en_vocal = True
        else:
            en_vocal = False

    return max(1, silabas)  # Toda palabra tiene al menos 1 sílaba


def calcular_metricas_texto(texto):
    """
    Devuelve las 5 features lingüísticas como lista para construir la matriz X.

    Versión simplificada: devuelve lista en lugar de dict para mayor
    eficiencia al construir el array de features.

    Returns:
        list[float]: [palabras_por_frase, silabas_por_palabra,
                      longitud_promedio_palabra, ratio_palabras_largas, inflesz]
    """
    texto = texto.strip()
    frases = re.split(r'[.!?]+', texto)
    frases = [f.strip() for f in frases if f.strip()]
    num_frases = max(len(frases), 1)  # Evitar división por cero

    palabras = re.findall(r'\b\w+\b', texto.lower())
    num_palabras = len(palabras)

    if num_palabras == 0:
        return [0, 0, 0, 0, 0]  # Vector de ceros para textos vacíos

    palabras_por_frase = num_palabras / num_frases
    total_silabas = sum(contar_silabas(p) for p in palabras)
    silabas_por_palabra = total_silabas / num_palabras
    longitud_promedio_palabra = sum(len(p) for p in palabras) / num_palabras

    # Proporción de palabras con más de 3 sílabas (proxy de vocabulario técnico)
    palabras_largas = sum(1 for p in palabras if contar_silabas(p) > 3)
    ratio_palabras_largas = palabras_largas / num_palabras

    # Índice INFLESZ: adaptación española del índice de Flesch
    # Valores altos = más fácil, valores bajos = más difícil
    inflesz = 206.84 - (60 * silabas_por_palabra) - (1.02 * palabras_por_frase)

    return [
        palabras_por_frase,
        silabas_por_palabra,
        longitud_promedio_palabra,
        ratio_palabras_largas,
        inflesz
    ]

print("✓ Funciones de features definidas")

In [ ]:
# Calcular features para todos los textos
print("Calculando features...")
# Aplicar las 5 métricas lingüísticas a cada fragmento del dataset
features_list = [calcular_metricas_texto(texto) for texto in df_final['texto']]
X = np.array(features_list)
y = df_final['nivel'].values

feature_names = [
    'palabras_por_frase',
    'silabas_por_palabra',
    'longitud_promedio_palabra',
    'ratio_palabras_largas',
    'inflesz'
]

print(f"✓ Features calculados: {X.shape[1]} features × {X.shape[0]} textos")

In [ ]:
# Estratificación para mantener proporciones de cada nivel en ambos splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {len(X_train)} textos")
print(f"Test set: {len(X_test)} textos")

In [ ]:
print("\nEntrenando RandomForest...")

clf = RandomForestClassifier(
    n_estimators=200,         # 200 árboles para mayor estabilidad que el baseline
    max_depth=20,             # Mayor profundidad que el baseline (10)
    min_samples_split=5,      # Evitar splits con muy pocas muestras
    class_weight='balanced',  # Compensar desbalanceo entre niveles
    random_state=42
)

clf.fit(X_train, y_train)
print("✓ Modelo entrenado")

## 9. Evaluar Modelo - VER MEJORA

In [ ]:
# Predicciones 
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("=" * 70)
print("📊 RESULTADOS DEL CLASIFICADOR")
print("=" * 70)
print(f"\n✓ ACCURACY: {accuracy:.2%}")
# Comparación respecto al baseline de 40% para mostrar la mejora
print(f"  Mejora desde 40% → {accuracy:.0%}")
print(f"  Incremento: +{(accuracy - 0.40) * 100:.0f} puntos porcentuales")
print("\n" + "=" * 70)

In [ ]:
# Reporte detallado
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
import plotly.figure_factory as ff

# Matriz de confusión interactiva: permite ver qué niveles se confunden entre sí.
# Errores entre niveles adyacentes (N2↔N3, N3↔N4) son esperables y justifican
# el paso a embeddings semánticos en el siguiente notebook.
cm = confusion_matrix(y_test, y_pred)
niveles = sorted(df_final['nivel'].unique())

fig = ff.create_annotated_heatmap(
    z=cm,
    x=[f'Nivel {n}' for n in niveles],
    y=[f'Nivel {n}' for n in niveles],
    colorscale='Blues',
    showscale=True
)

fig.update_layout(
    title='Matriz de Confusión',
    xaxis_title='Predicho',
    yaxis_title='Real'
)

fig.show()

In [ ]:
# Importancia de features

# Indica qué métricas lingüísticas son más determinantes para discriminar niveles
importancias = pd.DataFrame({
    'feature': feature_names,
    'importancia': clf.feature_importances_
}).sort_values('importancia', ascending=False)

fig = px.bar(importancias, x='importancia', y='feature',
             orientation='h',
             title='Importancia de Features en el Clasificador',
             labels={'importancia': 'Importancia', 'feature': 'Feature'})

fig.show()

print("\nImportancia de Features:")
for _, row in importancias.iterrows():
    print(f"  {row['feature']}: {row['importancia']:.3f}")

## 10. Guardar Modelo Mejorado

In [ ]:
import pickle

modelo_path = BASE_DIR / 'models' / 'clasificador_mejorado.pkl'
# Crear carpeta models si no existe
modelo_path.parent.mkdir(parents=True, exist_ok=True)

# NOTA: Este modelo usa features lingüísticas con más datos que el baseline.
# El modelo final del proyecto usa embeddings semánticos:
# models/clasificador_embeddings.pkl
with open(modelo_path, 'wb') as f:
    pickle.dump(clf, f)

print(f"✓ Modelo guardado en: {modelo_path}")
print(f"  Accuracy: {accuracy:.2%}")
print(f"  Textos de entrenamiento: {len(X_train)}")

## 11. Probar con Nuevos Textos

In [ ]:
# Textos de prueba representativos de N1 a N4
textos_prueba = [
    "El perro juega en el parque. Es un perro grande.",
    "La economía española presenta indicadores positivos según los últimos datos.",
    "El proceso de fotosíntesis implica la transformación de energía lumínica en energía química mediante complejos mecanismos bioquímicos.",
    "El Real Decreto establece las bases reguladoras para la concesión de subvenciones destinadas a entidades sin ánimo de lucro."
]

print("\n📝 PREDICCIONES EN TEXTOS DE PRUEBA")
print("=" * 80)

for i, texto in enumerate(textos_prueba, 1):
    # calcular_metricas_texto devuelve lista → envolver en array 2D para predict
    features = np.array([calcular_metricas_texto(texto)])
    nivel_pred = clf.predict(features)[0]
    proba = clf.predict_proba(features)[0]
    confianza = max(proba) * 100

    print(f"\nTexto {i}:")
    print(f"  '{texto}'")
    print(f"  → Nivel predicho: {nivel_pred}")
    print(f"  → Confianza: {confianza:.1f}%")

## ✅ RESUMEN FINAL

In [ ]:
print("\n" + "=" * 80)
print("✓ PROCESAMIENTO COMPLETADO CON ÉXITO")
print("=" * 80)
print(f"\n📊 ESTADÍSTICAS FINALES:")
print(f"  - Dataset generado: {len(df_final)} textos")
print(f"  - Distribución: {dict(df_final['nivel'].value_counts().sort_index())}")
print(f"  - CSV guardado en: {OUTPUT_CSV}")
print(f"\n🤖 MODELO:")
print(f"  - Accuracy: {accuracy:.2%}")
print(f"  - Mejora desde versión anterior: +{(accuracy - 0.40) * 100:.0f} puntos")
print(f"  - Modelo guardado en: {modelo_path}")
print(f"\n🎯 PRÓXIMOS PASOS:")
print(f"  1. Aplicar fragmentación adaptativa (codigo_fragmentacion_adaptativa.py)")
print(f"  2. Transición a embeddings semánticos (02_procesamiento_embeddings.ipynb)")
print(f"  3. Clustering K-means de usuarios (03_clustering_usuarios.py)")
print("\n" + "=" * 80)